# 01 - Ingestao de Dados

**Objetivo:** baixar os CSVs semestrais da ANP (serie historica de precos de combustiveis) e coletar dados externos de dolar e Brent.

**Fonte primaria:** [ANP - Serie Historica de Precos](https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/serie-historica-de-precos-de-combustiveis)

**Pipeline:**
1. Descobrir links de CSV na pagina da ANP
2. Filtrar apenas arquivos semestrais de 2004 a 2025
3. Fazer download dos arquivos validos para `data/raw`
4. Coletar cotacao do dolar (Banco Central)
5. Coletar preco do petroleo Brent (EIA/FRED)
6. (Opcional) Upload dos CSVs para AWS S3

**Regras do projeto:**
- Somente CSVs semestrais da ANP no intervalo 2004-2025.
- CSVs mensais e arquivos de 2026 da ANP sao ignorados.
- Upload para AWS S3 e opcional e nao e requisito para rodar o projeto.


In [1]:
import logging
import sys
from pathlib import Path

sys.path.insert(0, "..")

from src.ingestao import (
    descobrir_links_csv,
    download_todos_csvs,
    filtrar_links_csv,
    upload_todos_para_s3,
)
from src.scraping import obter_cotacao_dolar, obter_preco_brent, salvar_dados_externos
from src.utils import DATA_EXTERNAL, DATA_RAW

for logger_name in ("ingestao", "scraping"):
    logging.getLogger(logger_name).setLevel(logging.WARNING)


## 1.1 — Descobrir links de download na ANP

In [2]:
links = descobrir_links_csv()
links_validos, links_ignorados = filtrar_links_csv(
    links,
    ano_min=2004,
    ano_max=2025,
    apenas_semestrais=True,
)

print(f"Links CSV encontrados na ANP: {len(links)}")
print(f"Arquivos considerados (semestrais 2004-2025): {len(links_validos)}")
print(f"Arquivos ignorados (mensais, fora do recorte ou duplicados): {len(links_ignorados)}")
for link in links_validos[:5]:
    print(f"  {link['nome']}")
print("  ...")


Encontrados 200 arquivos CSV
  2021: 2º semestre de 2021
  2021: 1º semestre de 2021
  2020: 2º semestre de 2020
  2020: 1º semestre de 2020
  2019: 2º semestre de 2019
  ...


## 1.2 - Download dos CSVs
Apenas arquivos semestrais (2004-2025). Arquivos mensais e registros de 2026 da ANP sao ignorados.


In [3]:
antes = {p.name for p in DATA_RAW.glob("*.csv")}
arquivos = download_todos_csvs(
    DATA_RAW,
    ano_min=2004,
    ano_max=2025,
    apenas_semestrais=True,
)
depois = {p.name for p in DATA_RAW.glob("*.csv")}
novos = sorted(depois - antes)
pulados = max(len(arquivos) - len(novos), 0)

print("Diret?rio de destino: data/raw")
print(f"Arquivos considerados: {len(arquivos)}")
print(f"Arquivos baixados agora: {len(novos)}")
print(f"Arquivos j? existentes (pulados): {pulados}")
for nome in novos[:5]:
    tamanho_mb = (DATA_RAW / nome).stat().st_size / 1e6
    print(f"  {nome} ? {tamanho_mb:.1f} MB")
if len(novos) > 5:
    print("  ...")



44 arquivos baixados em c:\Users\leona\Desktop\TI\Combustivel Brasil Analytics\notebooks\..\data\raw
  2º_semestre_de_2021.csv — 39.7 MB
  1º_semestre_de_2021.csv — 57.9 MB
  2º_semestre_de_2020.csv — 2.4 MB
  1º_semestre_de_2020.csv — 86.7 MB
  2º_semestre_de_2019.csv — 89.0 MB


## 1.3 — Explorar metadados de um CSV

In [4]:
import pandas as pd

csvs = sorted(DATA_RAW.glob('*.csv'))
if csvs:
    amostra = pd.read_csv(csvs[0], sep=';', encoding='latin-1', nrows=5)
    print(f"Arquivo: {csvs[0].name}")
    print(f"Colunas: {list(amostra.columns)}")
    print(f"\nAmostra:")
    display(amostra)
else:
    print("Nenhum CSV encontrado. Execute o download primeiro.")

Arquivo: 1º_semestre_de_2004.csv
Colunas: ['ï»¿Regiao - Sigla', 'Estado - Sigla', 'Municipio', 'Revenda', 'CNPJ da Revenda', 'Nome da Rua', 'Numero Rua', 'Complemento', 'Bairro', 'Cep', 'Produto', 'Data da Coleta', 'Valor de Venda', 'Valor de Compra', 'Unidade de Medida', 'Bandeira']

Amostra:


,ï»¿Regiao - Sigla,Estado - Sigla,Municipio,Revenda,CNPJ da Revenda,Nome da Rua,Numero Rua,Complemento,Bairro,Cep,Produto,Data da Coleta,Valor de Venda,Valor de Compra,Unidade de Medida,Bandeira
0,SE,SP,GUARULHOS,AUTO POSTO SAKAMOTO LTDA,49.051.667/0001-02,RODOVIA PRESIDENTE DUTRA,S/N,"KM 210,5-SENT SP/RJ",BONSUCESSO,07178-580,GASOLINA,11/05/2004,"1,967","1,6623",R$ / litro,PETROBRAS DISTRIBUIDORA S.A.
1,SE,SP,GUARULHOS,AUTO POSTO SAKAMOTO LTDA,49.051.667/0001-02,RODOVIA PRESIDENTE DUTRA,S/N,"KM 210,5-SENT SP/RJ",BONSUCESSO,07178-580,ETANOL,11/05/2004,"0,899","0,6282",R$ / litro,PETROBRAS DISTRIBUIDORA S.A.
2,SE,SP,GUARULHOS,AUTO POSTO SAKAMOTO LTDA,49.051.667/0001-02,RODOVIA PRESIDENTE DUTRA,S/N,"KM 210,5-SENT SP/RJ",BONSUCESSO,07178-580,DIESEL,11/05/2004,"1,299","1,1704",R$ / litro,PETROBRAS DISTRIBUIDORA S.A.
3,SE,SP,SOROCABA,COMPETRO COMERCIO E DISTRIBUICAO DE DERIVADOS ...,00.003.188/0001-21,RUA HUMBERTO DE CAMPOS,306,NaN,JARDIM ZULMIRA,18061-000,GASOLINA,10/05/2004,"1,85","1,67",R$ / litro,BRANCA
4,SE,SP,SOROCABA,COMPETRO COMERCIO E DISTRIBUICAO DE DERIVADOS ...,00.003.188/0001-21,RUA HUMBERTO DE CAMPOS,306,NaN,JARDIM ZULMIRA,18061-000,ETANOL,10/05/2004,"0,78","0,48",R$ / litro,BRANCA


## 1.4 - Coleta de dados externos
Os dados externos (USD/BRL e Brent) podem ir alem de 2025, pois seguem a disponibilidade das fontes.


In [5]:
df_dolar = obter_cotacao_dolar()
print(f"Cotação do dólar: {len(df_dolar)} dias")
print(f"Período: {df_dolar['data'].min()} a {df_dolar['data'].max()}")
df_dolar.tail()

Cotação do dólar: 5595 dias
Período: 2004-01-02 00:00:00 a 2026-04-14 00:00:00


,data,cotacao_compra,cotacao_venda
5590,2026-04-08,5.0893,5.0899
5591,2026-04-09,5.0815,5.0821
5592,2026-04-10,5.0223,5.0229
5593,2026-04-13,5.0238,5.0244
5594,2026-04-14,4.9800,4.9806


In [6]:
df_brent = obter_preco_brent()
print(f"Preço Brent: {len(df_brent)} registros")
print(f"Período: {df_brent['data'].min()} a {df_brent['data'].max()}")
df_brent.tail()

Preço Brent: 9864 registros
Período: 1987-05-20 00:00:00 a 2026-04-02 00:00:00


,data,preco_brent_usd
9859,2026-03-27,121.47
9860,2026-03-30,121.88
9861,2026-03-31,126.69
9862,2026-04-01,119.56
9863,2026-04-02,127.61


In [7]:
paths = salvar_dados_externos(df_dolar, df_brent)
for nome, path in paths.items():
    print(f"{nome}: data/external/{Path(path).name}")


dolar: c:\Users\leona\Desktop\TI\Combustivel Brasil Analytics\notebooks\..\data\external\cotacao_dolar.parquet
brent: c:\Users\leona\Desktop\TI\Combustivel Brasil Analytics\notebooks\..\data\external\preco_brent.parquet


## 1.5 - Upload para AWS S3 (opcional)

Este passo e opcional e nao e requisito para rodar o projeto localmente.

Requer configuracao do arquivo `.env` com credenciais AWS.


In [8]:
# Descomente para executar o upload
# uris = upload_todos_para_s3(DATA_RAW)
# print(f"{len(uris)} arquivos enviados para S3")

---
**Próximo passo:** [02_etl_limpeza.ipynb](02_etl_limpeza.ipynb) — Limpeza e transformação dos dados